# KAT-VAD v2 — P3 / **retrain the lost phase-4 KIP-off checkpoints** (s2025, s2026)

Pre-registration: `core/docs/v2/PREREG_ADDENDUM.md` **§9 (Amendment 3, D12, J10)**.
On Drive, `DADA2000_orig_phase4/s2025` and `s2026` `stage2_kip_off/checkpoint_last.pt` are **mid-training snapshots**
(step 510 and 1530 of 2040): the finished weights never landed. This notebook retrains **only those two arms** with
phase 4's exact stage-2 KIP-off command, so E1 (`p3_e1.ipynb`) has three finished checkpoints.

| folder | holds | this notebook |
|---|---|---|
| `Thesis-V2/kat-vad/` | v2 code | imports + runs `core.train` from here |
| `Thesis-V2/outputs/v2_p4_retrain/s{seed}/stage2_kip_off/` | the retrained arms | **writes** |
| `Thesis/data/DADA2000_orig`, `Thesis/cache/{clip,knn}/DADA2000_orig` | T2 corpus, CLIP + KNN caches | reads |
| `Thesis/outputs/DADA2000_orig_phase4/` | phase-4 `config.yaml` (the reference) | reads only |

| step | what | GPU? |
|---|---|---|
| 1 | preflight + stage the corpus to VM disk | no |
| 2 | train s2025, s2026 (20 epochs = 2,040 steps each, in 5-epoch chunks, resumable) | **yes** |
| 3 | **verify on Drive**: config identical to phase 4, checkpoint `global_step` == 2040 == `metrics.jsonl`, sha256 == local | no |

**Nothing here reads DoTA.** Every flag is copied from `colab/DADA2000Origin/phase_4.ipynb` `train_cmd`; §3 proves it by
diffing `config.yaml` against phase 4's.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import sys
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # shared: data/, cache/, outputs/ (phase-4 reference)
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # v2: kat-vad/ (code), outputs/ (retrained arms)
os.environ['PROJECT_ROOT']       = DRIVE_V2
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_V2}/outputs'
V1_OUTPUTS = Path(DRIVE) / 'outputs'

os.environ['REPO'] = f'{DRIVE_V2}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v2 there'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time (see p3_e1 cell 2)
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
assert str(constants.OUTPUT_ROOT) == f'{DRIVE_V2}/outputs', constants.OUTPUT_ROOT

# --- phase 4's arm, restated from colab/DADA2000Origin/phase_4.ipynb (plan 6.2 of that campaign) ---
DATASET    = constants.DADA_ORIGIN_DATASET                 # 'DADA2000_orig'
KERNEL     = constants.DADA_ORIGIN_SCORE_HEAD_KERNEL       # 3  (C27)
TOPK_PCT   = constants.DADA_ORIGIN_MIL_TOPK_PCT            # 5  -> k = 4
NUM_EPOCHS = 20                                            # = 2,040 steps; also the LR-schedule horizon
FINAL_STEP = 2040
CHUNK_EPOCHS = 5
SEEDS      = (2025, 2026)                                  # s2024's finished checkpoint survived

T2        = constants.DATA_ROOT / DATASET
CLIP_DIR  = constants.CLIP_CACHE_DIR / DATASET
KNN_CACHE = constants.KNN_CACHE_DIR / DATASET / constants.KNN_CACHE_FILENAME
P4_REF    = V1_OUTPUTS / f'{DATASET}_phase4'               # phase-4 config.yaml per seed (reference)
OUT       = constants.OUTPUT_ROOT / 'v2_p4_retrain'        # s{seed}/stage2_kip_off
LOCAL     = Path('/content/rt')                            # VM-local: runs + staged inputs
LOCAL.mkdir(parents=True, exist_ok=True)
for name, path in (('corpus', T2), ('clip', CLIP_DIR), ('knn', KNN_CACHE), ('phase4', P4_REF), ('out', OUT)):
    print(f'  {name:7s} {path}   {"OK" if path.exists() else "MISSING (out is created)"}')

In [ ]:
%%bash
# Same pins as phase_4.ipynb (C7: transformers internals on the text path).
# Restart the runtime after this cell if pip printed a transformers downgrade, then re-run from cell 1.
pip install -q "transformers==4.56.*" av einops faiss-cpu
python -c "import transformers, av, einops, faiss; print('transformers', transformers.__version__, '| av/einops/faiss OK')"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- training will be unusably slow'
df -h /content | tail -1

## 1. Preflight + stage inputs to VM disk (bit-identical copies; C2 untouched)

In [ ]:
import hashlib
import json
import shutil
import subprocess
import time

import torch

addendum = (REPO / 'core/docs/v2/PREREG_ADDENDUM.md').read_text(encoding='utf-8')
assert 'Amendment 3' in addendum and 'J10' in addendum, f'{REPO} predates Amendment 3 -- re-upload the v2 code'
for seed in SEEDS:
    ref = P4_REF / f's{seed}' / 'stage2_kip_off' / 'config.yaml'
    assert ref.is_file(), f'missing phase-4 reference {ref}'
for path in (T2, CLIP_DIR, KNN_CACHE):
    assert path.exists(), f'missing {path}'


def stage(src, name):
    dst = LOCAL / 'stage' / name
    if dst.exists():
        print(f'  {name:14s} already staged')
        return dst
    t0 = time.time()
    if src.is_dir():
        shutil.copytree(src, dst)
    else:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
    files = [f for f in dst.rglob('*') if f.is_file()] if dst.is_dir() else [dst]
    print(f'  {name:14s} {len(files):5d} files, {sum(f.stat().st_size for f in files) / 2**20:8.1f} MiB '
          f'in {time.time() - t0:5.1f}s')
    return dst


L_T2 = stage(T2, 'corpus')
L_CLIP = stage(CLIP_DIR, 'clip')
L_KNN = stage(KNN_CACHE, 'knn_cache.npz')

## 2. Train s2025 and s2026 (stage 2, KIP off)

Chunked like phase 4: 5 epochs per invocation, `--resume` in between, synced to Drive after every chunk, so a
disconnect costs ≤ 5 epochs — **re-run this cell after a disconnect** and it continues.

The skip/resume decision reads the **checkpoint's own `global_step`**, not `metrics.jsonl` — that shortcut is exactly
how the originals were lost (Amendment 3).

In [ ]:
ENV = dict(os.environ)


def run(cmd):
    # Stream the child's output here (Colab shows nothing of a bare subprocess).
    tail = []
    with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
        for line in proc.stdout:
            tail = (tail + [line])[-40:]
            if 'global_step' not in line:          # per-batch metric lines stay in metrics.jsonl
                print(line, end='')
    if proc.returncode:
        print(''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd)}')


def train_cmd(out_dir, seed):
    # phase_4.ipynb train_cmd(stage=2, kip_on=False), flag for flag.
    return [sys.executable, '-m', 'core.train',
            '--set', 'train.stage=2',
            '--set', f'train.seed={seed}',
            '--set', f'train.num_epochs={NUM_EPOCHS}',
            '--set', 'train.amp=true',
            '--set', f'data.dataset={DATASET}',
            '--set', f'model.score_head_kernel={KERNEL}',
            '--set', f'loss.mil_topk_pct={TOPK_PCT}',
            '--data-dir', L_T2, '--clip-dir', L_CLIP,
            '--knn-cache', L_KNN,
            '--output-dir', out_dir,
            '--set', 'kip.enabled=false']


def ckpt_step(path):
    if not path.exists():
        return 0
    payload = torch.load(path, map_location='cpu', weights_only=False)
    return int(payload.get('global_step', 0))


def epochs_done(run_dir):
    path = run_dir / 'metrics.jsonl'
    if not path.exists():
        return 0
    lines = [l for l in path.read_text().splitlines() if l.strip()]
    return json.loads(lines[-1])['epoch'] + 1 if lines else 0


def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        for block in iter(lambda: fh.read(1 << 24), b''):
            h.update(block)
    return h.hexdigest()


def sync_to_drive(local, drive):
    drive.mkdir(parents=True, exist_ok=True)
    for name in ('checkpoint_last.pt', 'metrics.jsonl', 'config.yaml'):
        src = local / name
        if src.exists():
            shutil.copy2(src, drive / name)
            with open(drive / name, 'rb+') as fh:     # ask FUSE to push the bytes now (C10)
                os.fsync(fh.fileno())


def restore_from_drive(drive, local):
    # Resume only from a Drive checkpoint whose step matches its metrics (a torn sync is discarded).
    if not (drive / 'checkpoint_last.pt').exists():
        return
    local.mkdir(parents=True, exist_ok=True)
    if (local / 'checkpoint_last.pt').exists():
        return
    step = ckpt_step(drive / 'checkpoint_last.pt')
    lines = [l for l in (drive / 'metrics.jsonl').read_text().splitlines() if l.strip()]
    if not lines or json.loads(lines[-1])['global_step'] != step:
        print(f'  {drive}: checkpoint step {step} != metrics -- NOT resuming from it; starting over')
        return
    for name in ('checkpoint_last.pt', 'metrics.jsonl', 'config.yaml'):
        shutil.copy2(drive / name, local / name)


for seed in SEEDS:
    drive_dir = OUT / f's{seed}' / 'stage2_kip_off'
    local = LOCAL / 'runs' / f's{seed}' / 'stage2_kip_off'
    if ckpt_step(drive_dir / 'checkpoint_last.pt') == FINAL_STEP:
        print(f'seed {seed}: finished checkpoint already on Drive (step {FINAL_STEP}) -- skipping')
        continue
    restore_from_drive(drive_dir, local)
    while (done := epochs_done(local)) < NUM_EPOCHS:
        cmd = train_cmd(local, seed)
        if done:
            cmd += ['--resume', local / 'checkpoint_last.pt']
        cmd += ['--stop-after-epochs', min(CHUNK_EPOCHS, NUM_EPOCHS - done)]
        print(f'=== seed {seed}: epochs {done} -> {min(done + CHUNK_EPOCHS, NUM_EPOCHS)} of {NUM_EPOCHS}')
        run(cmd)
        sync_to_drive(local, drive_dir)
    assert ckpt_step(local / 'checkpoint_last.pt') == FINAL_STEP, 'local checkpoint is not the final step'
    sync_to_drive(local, drive_dir)
    print(f'seed {seed}: trained to step {FINAL_STEP}, synced')

## 3. Verify what is on Drive (the check phase 4 never made)

Reads the **Drive** copies back: config identical to phase 4's (same seed), checkpoint `global_step` == 2040 ==
`metrics.jsonl` last step (J10), and sha256 equal to the VM-local file when that still exists.

In [ ]:
import difflib

ok = True
for seed in SEEDS:
    drive_dir = OUT / f's{seed}' / 'stage2_kip_off'
    ref = (P4_REF / f's{seed}' / 'stage2_kip_off' / 'config.yaml').read_text().splitlines()
    new = (drive_dir / 'config.yaml').read_text().splitlines()
    diff = list(difflib.unified_diff(ref, new, 'phase4', 'retrain', lineterm=''))
    step = ckpt_step(drive_dir / 'checkpoint_last.pt')
    last = json.loads([l for l in (drive_dir / 'metrics.jsonl').read_text().splitlines() if l.strip()][-1])
    local_ck = LOCAL / 'runs' / f's{seed}' / 'stage2_kip_off' / 'checkpoint_last.pt'
    same = sha256(local_ck) == sha256(drive_dir / 'checkpoint_last.pt') if local_ck.exists() else None
    print(f'seed {seed}: config diff lines {len(diff)} | ckpt step {step} | metrics last step '
          f'{last["global_step"]} | sha256 local==drive {same}')
    if diff:
        print('\n'.join(diff))
    ok &= not diff and step == last['global_step'] == FINAL_STEP and same is not False
assert ok, 'retrained arms are NOT verified -- do not run E1 on them; send this output to Claude'
print('\nVERIFIED: both retrained checkpoints are finished, on Drive, with phase 4\'s exact config.')
print('Next: colab/v2/p3_e1.ipynb (it picks these up automatically).')